In [0]:
from pyspark.sql import functions as F

In [0]:
df_bronze_satisfacao = spark.read.table("projeto.bronze.pesquisa_satisfacao")

SEI QUE AS NOTAS DEVEM SER DE 1 A 5 O ID

In [0]:
display(df_bronze_satisfacao.limit(5))

Primeiramente irei fazer um cast dos meus dados

In [0]:
df_bronze_satisfacao = (
    df_bronze_satisfacao
    .withColumn("id_chamado", F.col("id_chamado").cast("int"))
    .withColumn("id_pesquisa", F.col("id_pesquisa").cast("int"))
    .withColumn("nota_atendimento", F.col("nota_atendimento").cast("int"))
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp"))
)

In [0]:
from pyspark.sql import functions as F

df = spark.table("projeto.bronze.pesquisa_satisfacao") 

total_linhas = df.count()
total_chamados_unicos = df.select("id_chamado").distinct().count()
total_pesquisas_unicas = df.select("id_pesquisa").distinct().count()
qtd_duplicatas_chamado = total_linhas - total_chamados_unicos

qtd_nulos_nota = df.filter(F.col("nota_atendimento").isNull()).count()

df_outliers = df.filter(
    (F.col("nota_atendimento") < 1) | 
    (F.col("nota_atendimento") > 5)
)
qtd_outliers = df_outliers.count()
df_dados_incompletos = df.filter(
    F.col("id_chamado").isNull() |
    F.col("id_pesquisa").isNull() |
    F.col("nota_atendimento").isNull() |
    F.col("ingestion_timestamp").isNull()
)

print("RESUMO DE QUALIDADE DE DADOS")
print(f"Total de Linhas:{total_linhas}")
print(f"Chamados Únicos:{total_chamados_unicos}")
print(f"IDs Pesquisa Únicos:{total_pesquisas_unicas}")
print(f"Duplicatas de Chamado:{qtd_duplicatas_chamado}")
print(f"Notas Nulas:{qtd_nulos_nota}")
print(f"Notas Outliers (<1 ou >5):{qtd_outliers}")
print(f"Registros Incompleto{df_dados_incompletos.count()}")

if df_dados_incompletos.count() > 0:
    print("\nVisualizando amostra de dados incompletos:")
    display(df_dados_incompletos.limit(5))

if qtd_outliers > 0:
    print("\nVisualizando amostra de outliers:")
    display(df_outliers.limit(5))